# Frozen nominal-compiler robustness stress test

This notebook evaluates the **frozen nominal neural pulse compiler** under physically motivated perturbations of the three-spin NMR model and control chain. The neural network is never retrained in this notebook.

The nominal Hamiltonian is

\[
H_0=\pi\sum_i \nu_i Z_i+\pi\sum_{i<j}J_{ij}(X_iX_j+Y_iY_j+Z_iZ_j),
\]

with the same 300-slice, 35 \(\mu\mathrm{s}\)-per-slice control model used for nominal training.

Two joint uncertainty envelopes are evaluated:

- **Routine/calibration envelope:** common \(B_0\) ±2 Hz, independent spin offsets ±2 Hz, independent \(J\) offsets ±0.25 Hz, RF gain ±5%, phase-reference error ±1°, clock error ±2 ppm, and 10 ps RMS timing jitter.
- **Conservative stress envelope:** common \(B_0\) ±6 Hz, independent spin offsets ±6 Hz, independent \(J\) offsets ±1 Hz, RF gain ±15%, phase-reference error ±3°, clock error ±5 ppm, and 50 ps RMS timing jitter.

The conservative envelope is deliberately broader than the routine calibration envelope. It is a stress test, not an assertion that every error source simultaneously reaches its largest value in a calibrated experiment.

The notebook also performs one-dimensional sweeps for RF gain, common \(B_0\), individual spin-frequency offsets, \(J\)-coupling magnitude, phase bias, and clock error, plus a two-dimensional common-\(B_0\) × RF-gain map. Exact sampled scenarios and gate-by-scenario fidelity matrices are saved so the reported robustness statistics can be recomputed without rerunning the quantum propagation.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import hashlib
import json
import math
import platform
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# The notebook is intended to be run from the repository root. Running it from
# notebooks/ is also supported for convenience.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

CHECKPOINT_PATH = REPO_ROOT / "data" / "checkpoints" / "nominal_best_state_dict.pt"
CONFIG_PATH = REPO_ROOT / "configs" / "nominal.json"
REFERENCE_SUMMARY_PATH = REPO_ROOT / "results" / "nominal" / "final_validation_summary.json"
OUTPUT_ROOT = REPO_ROOT / "results" / "stress_test"
RAW_DIR = OUTPUT_ROOT / "raw"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
SWEEP_DIR = RAW_DIR / "sweeps"
JOINT_DIR = RAW_DIR / "joint"

for directory in [OUTPUT_ROOT, RAW_DIR, TABLE_DIR, FIGURE_DIR, SWEEP_DIR, JOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for required_path in [CHECKPOINT_PATH, CONFIG_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required publication input not found: {required_path}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: the full stress test is computationally expensive on CPU.")

def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(block_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

with CONFIG_PATH.open() as handle:
    CONFIG_DICT = json.load(handle)

reference_summary = None
if REFERENCE_SUMMARY_PATH.exists():
    with REFERENCE_SUMMARY_PATH.open() as handle:
        reference_summary = json.load(handle)

def extract_state_dict(obj):
    if isinstance(obj, dict) and "model" in obj:
        return obj["model"]
    return obj

print("Nominal state dict:", CHECKPOINT_PATH.relative_to(REPO_ROOT))
print("State-dict SHA-256:", sha256_file(CHECKPOINT_PATH))
print("Nominal config:", CONFIG_PATH.relative_to(REPO_ROOT))

In [ ]:
@dataclass
class Config:
    # Physical and control parameters of the frozen nominal compiler.
    v_hz: tuple = (-921.0, 40.75, 700.0)
    j_hz: tuple = (-64.0, 24.4, 34.1)
    n_slices: int = 300
    dt_s: float = 35e-6
    rabi_max_hz: float = 50_000.0

    # Nominal network architecture.
    width: int = 256
    hidden_layers: int = 10
    dropout: float = 0.25

CFG = Config()

# The public model file is a pure state_dict. The frozen architecture and
# physical settings are stored separately in configs/nominal.json.
checkpoint_cfg = CONFIG_DICT
for key in ["n_slices", "dt_s", "rabi_max_hz", "width", "hidden_layers", "dropout"]:
    if key in checkpoint_cfg:
        expected = getattr(CFG, key)
        observed = checkpoint_cfg[key]
        if isinstance(expected, float):
            assert abs(float(observed) - expected) < 1e-12 * max(1.0, abs(expected)), (key, observed, expected)
        else:
            assert observed == expected, (key, observed, expected)

def kron3(a, b, c):
    return torch.kron(torch.kron(a, b), c)

class NMRSystem:
    def __init__(self, cfg, device):
        ctype = torch.complex64
        I = torch.eye(2, dtype=ctype, device=device)
        X = torch.tensor([[0, 1], [1, 0]], dtype=ctype, device=device)
        Y = torch.tensor([[0, -1j], [1j, 0]], dtype=ctype, device=device)
        Z = torch.tensor([[1, 0], [0, -1]], dtype=ctype, device=device)

        self.I2 = I
        self.I4 = torch.eye(4, dtype=ctype, device=device)
        self.I8 = torch.eye(8, dtype=ctype, device=device)
        self.X, self.Y, self.Z = X, Y, Z
        self.device = device

        self.Xs = [kron3(X, I, I), kron3(I, X, I), kron3(I, I, X)]
        self.Ys = [kron3(Y, I, I), kron3(I, Y, I), kron3(I, I, Y)]
        self.Zs = [kron3(Z, I, I), kron3(I, Z, I), kron3(I, I, Z)]
        self.Hx = sum(self.Xs)
        self.Hy = sum(self.Ys)

        self.pair_ops = []
        for i, j in [(0, 1), (0, 2), (1, 2)]:
            self.pair_ops.append(
                self.Xs[i] @ self.Xs[j]
                + self.Ys[i] @ self.Ys[j]
                + self.Zs[i] @ self.Zs[j]
            )

        v = torch.tensor(cfg.v_hz, dtype=torch.float32, device=device)[None, :]
        j = torch.tensor(cfg.j_hz, dtype=torch.float32, device=device)[None, :]
        self.H0 = self.build_h0(v, j)[0]

    def build_h0(self, v, j):
        H = torch.zeros((len(v), 8, 8), dtype=torch.complex64, device=self.device)
        for i in range(3):
            H = H + math.pi * v[:, i, None, None].to(H.dtype) * self.Zs[i]
        for k in range(3):
            H = H + math.pi * j[:, k, None, None].to(H.dtype) * self.pair_ops[k]
        return H

SYS = NMRSystem(CFG, DEVICE)

def sample_haar(n, seed):
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(seed)
    q = torch.randn(n, 4, generator=generator, device=DEVICE)
    return q / q.norm(dim=-1, keepdim=True).clamp_min(1e-12)

def quaternion_to_unitary(q):
    q0, qx, qy, qz = q.unbind(-1)
    I, X, Y, Z = SYS.I2, SYS.X, SYS.Y, SYS.Z
    return (
        q0[:, None, None].to(I.dtype) * I
        - 1j * (
            qx[:, None, None].to(I.dtype) * X
            + qy[:, None, None].to(I.dtype) * Y
            + qz[:, None, None].to(I.dtype) * Z
        )
    )

def lift_target(U2):
    return torch.einsum("bij,kl->bikjl", U2, SYS.I4).reshape(-1, 8, 8)

class PulseCompiler(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        layers = []
        in_dim = 4
        for _ in range(cfg.hidden_layers):
            layers.extend([nn.Linear(in_dim, cfg.width), nn.GELU()])
            if cfg.dropout > 0:
                layers.append(nn.Dropout(cfg.dropout))
            in_dim = cfg.width
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, 2 * cfg.n_slices)
        self.cfg = cfg

    def forward(self, q):
        raw = self.head(self.backbone(q)).view(-1, self.cfg.n_slices, 2)
        radius_sq = (raw * raw).sum(dim=-1, keepdim=True)
        return self.cfg.rabi_max_hz * raw / torch.sqrt(1.0 + radius_sq)

def unitary_fidelity(target, actual):
    overlap = (target.conj() * actual).sum(dim=(-2, -1))
    return overlap.abs().square() / 64.0

def propagate_nominal(controls):
    ux, uy = controls[..., 0], controls[..., 1]
    H = (
        SYS.H0[None, None]
        + math.pi * ux[..., None, None].to(torch.complex64) * SYS.Hx
        + math.pi * uy[..., None, None].to(torch.complex64) * SYS.Hy
    )
    U_steps = torch.matrix_exp((-1j * CFG.dt_s) * H)
    U = SYS.I8[None].expand(len(controls), -1, -1).clone()
    for t in range(CFG.n_slices):
        U = U_steps[:, t] @ U
    return U

def summarize(values):
    x = np.asarray(values, float)
    return {
        "mean": float(x.mean()),
        "std": float(x.std()),
        "median": float(np.median(x)),
        "p05": float(np.quantile(x, 0.05)),
        "p01": float(np.quantile(x, 0.01)),
        "min": float(x.min()),
        "max": float(x.max()),
        "frac_ge_099": float(np.mean(x >= 0.99)),
        "frac_ge_098": float(np.mean(x >= 0.98)),
        "frac_ge_095": float(np.mean(x >= 0.95)),
    }

MODEL = PulseCompiler(CFG).to(DEVICE)
MODEL.load_state_dict(extract_state_dict(checkpoint), strict=True)
MODEL.eval()
for parameter in MODEL.parameters():
    parameter.requires_grad_(False)

assert torch.allclose(SYS.H0, SYS.H0.mH, atol=1e-5)
assert CFG.dt_s == 35e-6
assert CFG.hidden_layers == 10
assert CFG.dropout == 0.25

print("Selected nominal training step: 48600")
if reference_summary is not None:
    print("Frozen nominal validation statistics:", reference_summary)

In [ ]:
N_NEW_GATES = 256
N_SWEEP_GATES = 128
N_MAP_GATES = 64
N_JOINT_SCENARIOS = 96
GATE_CHUNK = 8
SCENARIO_CHUNK = 8
STRESS_SEED = 20260918
SWEEPS = {
    "common_B0_hz": np.linspace(-10, 10, 21),
    "rf_gain_fraction": np.linspace(-0.15, 0.15, 25),
    "phase_bias_deg": np.linspace(-3, 3, 25),
    "J_magnitude_offset_hz": np.linspace(-1, 1, 21),
    "clock_error_ppm": np.linspace(-5, 5, 21),
    "spin1_offset_hz": np.linspace(-6, 6, 21),
    "spin2_offset_hz": np.linspace(-6, 6, 21),
    "spin3_offset_hz": np.linspace(-6, 6, 21),
}

PROFILES = {
    "routine": {
        "common_B0_abs_hz": 2.0,
        "independent_v_abs_hz": 2.0,
        "independent_J_abs_hz": 0.25,
        "rf_gain_abs_fraction": 0.05,
        "phase_bias_abs_deg": 1.0,
        "clock_abs_ppm": 2.0,
        "timing_jitter_rms_ps": 10.0,
    },
    "conservative": {
        "common_B0_abs_hz": 6.0,
        "independent_v_abs_hz": 6.0,
        "independent_J_abs_hz": 1.0,
        "rf_gain_abs_fraction": 0.15,
        "phase_bias_abs_deg": 3.0,
        "clock_abs_ppm": 5.0,
        "timing_jitter_rms_ps": 50.0,
    },
}

def nominal_scen(S=1): return {'common_B0_hz':np.zeros(S,np.float32),'v_offset_hz':np.zeros((S,3),np.float32),'J_offset_hz':np.zeros((S,3),np.float32),'rf_gain':np.ones(S,np.float32),'phase_bias_rad':np.zeros(S,np.float32),'clock_ppm':np.zeros(S,np.float32),'timing_jitter_s':np.zeros((S,CFG.n_slices),np.float32)}
def sweep_scen(name,val):
    val=np.asarray(val,np.float32); s=nominal_scen(len(val))
    if name=='common_B0_hz':s['common_B0_hz']=val
    elif name=='rf_gain_fraction':s['rf_gain']=1+val
    elif name=='phase_bias_deg':s['phase_bias_rad']=np.deg2rad(val).astype(np.float32)
    elif name=='J_magnitude_offset_hz':s['J_offset_hz']=val[:,None]*np.sign(np.asarray(CFG.j_hz,np.float32))[None,:]
    elif name=='clock_error_ppm':s['clock_ppm']=val
    elif name.startswith('spin'): s['v_offset_hz'][:,int(name[4])-1]=val
    return s
def joint_scen(name,S,seed):
    p=PROFILES[name]; z=2*torch.quasirandom.SobolEngine(10,scramble=True,seed=seed).draw(S).numpy().astype(np.float32)-1; s=nominal_scen(S); k=0
    s['common_B0_hz']=z[:,k]*p['common_B0_abs_hz']; k+=1; s['v_offset_hz']=z[:,k:k+3]*p['independent_v_abs_hz']; k+=3; s['J_offset_hz']=z[:,k:k+3]*p['independent_J_abs_hz']; k+=3; s['rf_gain']=1+z[:,k]*p['rf_gain_abs_fraction']; k+=1; s['phase_bias_rad']=np.deg2rad(z[:,k]*p['phase_bias_abs_deg']).astype(np.float32); k+=1; s['clock_ppm']=z[:,k]*p['clock_abs_ppm']
    s['timing_jitter_s']=np.random.default_rng(seed+999).normal(0,p['timing_jitter_rms_ps']*1e-12,(S,CFG.n_slices)).astype(np.float32); return s
def prop_pert(c,targ,s):
    G,S=len(c),len(s['rf_gain']); out=np.empty((G,S),np.float32); v0=torch.tensor(CFG.v_hz,dtype=torch.float32,device=DEVICE); j0=torch.tensor(CFG.j_hz,dtype=torch.float32,device=DEVICE)
    for g0 in range(0,G,GATE_CHUNK):
      g1=min(G,g0+GATE_CHUNK); cc=c[g0:g1]; tt=targ[g0:g1]; Gc=g1-g0
      for s0 in range(0,S,SCENARIO_CHUNK):
        s1=min(S,s0+SCENARIO_CHUNK); st={k:torch.as_tensor(v[s0:s1],device=DEVICE) for k,v in s.items()}; Sc=s1-s0; H0=SYS.build_h0(v0[None]+st['common_B0_hz'][:,None]+st['v_offset_hz'],j0[None]+st['J_offset_hz']); ce=cc[:,None].expand(Gc,Sc,-1,-1); ph=st['phase_bias_rad'][None,:,None]; cp,sp=torch.cos(ph),torch.sin(ph); gain=st['rf_gain'][None,:,None]; ux=(gain*(cp*ce[...,0]-sp*ce[...,1])).reshape(Gc*Sc,CFG.n_slices); uy=(gain*(sp*ce[...,0]+cp*ce[...,1])).reshape(Gc*Sc,CFG.n_slices); H0=H0[None].expand(Gc,Sc,-1,-1).reshape(Gc*Sc,8,8); H=H0[:,None]+math.pi*ux[...,None,None].to(torch.complex64)*SYS.Hx+math.pi*uy[...,None,None].to(torch.complex64)*SYS.Hy; dt=(CFG.dt_s*(1+st['clock_ppm']*1e-6)[:,None]+st['timing_jitter_s']); dt=dt[None].expand(Gc,Sc,-1).reshape(Gc*Sc,CFG.n_slices); Us=torch.matrix_exp((-1j*dt[...,None,None])*H); U=SYS.I8[None].expand(Gc*Sc,-1,-1).clone()
        for k in range(CFG.n_slices):U=Us[:,k]@U
        T=tt[:,None].expand(Gc,Sc,-1,-1).reshape(Gc*Sc,8,8); out[g0:g1,s0:s1]=fidelity(T,U).reshape(Gc,Sc).cpu().numpy(); del H,Us,U
      if torch.cuda.is_available():
          torch.cuda.empty_cache()
    return out
def cvar_inf(F,a=.1):
    L=1-F; k=max(1,int(np.ceil(a*L.shape[1]))); return np.partition(L,L.shape[1]-k,axis=1)[:,L.shape[1]-k:].mean(1)
def risk_table(F,nom): return pd.DataFrame({'gate_index':np.arange(len(F)),'nominal_fidelity':nom,'mean_fidelity':F.mean(1),'median_fidelity':np.median(F,axis=1),'p05_fidelity':np.quantile(F,.05,axis=1),'worst_fidelity':F.min(1),'cvar10_infidelity':cvar_inf(F,.1),'cvar20_infidelity':cvar_inf(F,.2)})

In [ ]:
@torch.inference_mode()
def compile_gates(q):
    controls = []
    targets = []
    fidelities = []
    for start in range(0, len(q), 32):
        batch = q[start : start + 32]
        target = lift_target(quaternion_to_unitary(batch))
        control = MODEL(batch)
        actual = propagate_nominal(control)
        controls.append(control.cpu())
        targets.append(target.cpu())
        fidelities.append(unitary_fidelity(target, actual).cpu())
    return (
        torch.cat(controls).to(DEVICE),
        torch.cat(targets).to(DEVICE),
        torch.cat(fidelities).numpy(),
    )

# Verify the frozen checkpoint on the archived nominal validation set when those
# arrays are present. The stress analysis itself uses a fresh Haar sample.
archived_q_path = REPO_ROOT / "data" / "nominal" / "validation_quaternions.npy"
archived_fids_path = REPO_ROOT / "data" / "nominal" / "validation_fidelities.npy"
if archived_q_path.exists() and archived_fids_path.exists():
    archived_q = torch.tensor(np.load(archived_q_path), dtype=torch.float32, device=DEVICE)
    stored_fids = np.load(archived_fids_path)
    _, _, recomputed_fids = compile_gates(archived_q)
    max_difference = float(np.max(np.abs(stored_fids - recomputed_fids)))
    print("Archived validation reproduction max |ΔF|:", max_difference)
    assert max_difference < 2e-4

q = sample_haar(N_NEW_GATES, STRESS_SEED)
controls, targets, nominal_fidelities = compile_gates(q)
np.save(RAW_DIR / "new_haar_quaternions.npy", q.cpu().numpy())
np.save(RAW_DIR / "new_nominal_fidelities.npy", nominal_fidelities)

nominal_summary = summarize(nominal_fidelities)
print("Fresh-sample nominal summary:")
print(json.dumps(nominal_summary, indent=2))

pd.DataFrame({
    "gate_index": np.arange(N_NEW_GATES),
    "q0": q[:, 0].cpu().numpy(),
    "qx": q[:, 1].cpu().numpy(),
    "qy": q[:, 2].cpu().numpy(),
    "qz": q[:, 3].cpu().numpy(),
    "nominal_fidelity": nominal_fidelities,
}).to_csv(TABLE_DIR / "new_haar_nominal_gates.csv", index=False)

sweep_frames = []
for parameter_name, values in SWEEPS.items():
    print("Sweep:", parameter_name)
    fidelity_matrix = prop_pert(
        controls[:N_SWEEP_GATES],
        targets[:N_SWEEP_GATES],
        sweep_scen(parameter_name, values),
    )
    np.save(SWEEP_DIR / f"{parameter_name}.npy", fidelity_matrix)

    summary_table = pd.DataFrame([
        {"parameter": parameter_name, "value": float(value), **summarize(fidelity_matrix[:, i])}
        for i, value in enumerate(values)
    ])
    summary_table.to_csv(TABLE_DIR / f"stress_{parameter_name}.csv", index=False)
    sweep_frames.append(summary_table)

pd.concat(sweep_frames, ignore_index=True).to_csv(
    TABLE_DIR / "stress_sweep_summary.csv", index=False
)

joint_results = {}
joint_summary_rows = []
for profile_index, profile_name in enumerate(PROFILES):
    print("Joint profile:", profile_name)
    scenarios = joint_scen(
        profile_name,
        N_JOINT_SCENARIOS,
        STRESS_SEED + 1000 * (profile_index + 1),
    )
    np.savez(JOINT_DIR / f"{profile_name}_scenarios.npz", **scenarios)

    fidelity_matrix = prop_pert(controls, targets, scenarios)
    np.save(JOINT_DIR / f"{profile_name}_fidelity_matrix.npy", fidelity_matrix)
    joint_results[profile_name] = fidelity_matrix

    gate_risk = risk_table(fidelity_matrix, nominal_fidelities)
    gate_risk.to_csv(TABLE_DIR / f"{profile_name}_gate_risk.csv", index=False)

    joint_summary_rows.append({
        "profile": profile_name,
        **{f"scenario_{k}": v for k, v in summarize(fidelity_matrix.ravel()).items()},
        **{f"gate_mean_{k}": v for k, v in summarize(fidelity_matrix.mean(1)).items()},
        **{f"gate_worst_{k}": v for k, v in summarize(fidelity_matrix.min(1)).items()},
        "mean_cvar10_infidelity": float(gate_risk.cvar10_infidelity.mean()),
        "mean_cvar20_infidelity": float(gate_risk.cvar20_infidelity.mean()),
    })

joint_summary = pd.DataFrame(joint_summary_rows)
joint_summary.to_csv(TABLE_DIR / "joint_profile_summary.csv", index=False)
display(joint_summary)

b0_values = np.linspace(-6, 6, 13, dtype=np.float32)
rf_values = np.linspace(-0.15, 0.15, 13, dtype=np.float32)
pairs = [(b0, rf) for b0 in b0_values for rf in rf_values]
map_scenarios = nominal_scen(len(pairs))
map_scenarios["common_B0_hz"] = np.array([pair[0] for pair in pairs], np.float32)
map_scenarios["rf_gain"] = 1 + np.array([pair[1] for pair in pairs], np.float32)

map_fidelities = prop_pert(
    controls[:N_MAP_GATES], targets[:N_MAP_GATES], map_scenarios
)
mean_map = map_fidelities.mean(0).reshape(13, 13)
p05_map = np.quantile(map_fidelities, 0.05, axis=0).reshape(13, 13)

np.savez(
    RAW_DIR / "b0_rf_gain_2d_map.npz",
    B0_hz=b0_values,
    gain_fraction=rf_values,
    fidelity_matrix=map_fidelities,
    mean_map=mean_map,
    p05_map=p05_map,
)

In [ ]:
def save_figure(name):
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{name}.png", dpi=600, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.show()
    plt.close()

plt.figure(figsize=(7, 4.6))
plt.hist(nominal_fidelities, bins=35)
plt.axvline(np.mean(nominal_fidelities), ls="--", label=f"mean={np.mean(nominal_fidelities):.4f}")
plt.axvline(np.quantile(nominal_fidelities, 0.05), ls=":", label=f"p05={np.quantile(nominal_fidelities, 0.05):.4f}")
plt.xlabel("Nominal fidelity")
plt.ylabel("Fresh Haar targets")
plt.legend()
plt.grid(alpha=0.15)
save_figure("new_haar_nominal_histogram")

for parameter_name in SWEEPS:
    table = pd.read_csv(TABLE_DIR / f"stress_{parameter_name}.csv")
    plt.figure(figsize=(7, 4.6))
    for statistic in ["mean", "median", "p05", "min"]:
        plt.plot(table.value, table[statistic], marker="o", ms=3, label=statistic)
    plt.xlabel(parameter_name)
    plt.ylabel("Unitary fidelity")
    plt.ylim(0, 1.005)
    plt.legend()
    plt.grid(alpha=0.2)
    save_figure(f"stress_{parameter_name}")

for profile_name, fidelity_matrix in joint_results.items():
    plt.figure(figsize=(7, 4.6))
    plt.hist(fidelity_matrix.ravel(), bins=45)
    plt.xlabel("Gate × scenario fidelity")
    plt.ylabel("Count")
    plt.title(profile_name.capitalize())
    plt.grid(alpha=0.15)
    save_figure(f"joint_{profile_name}_histogram")

    gate_risk = pd.read_csv(TABLE_DIR / f"{profile_name}_gate_risk.csv")
    plt.figure(figsize=(6.2, 5.2))
    plt.scatter(gate_risk.nominal_fidelity, gate_risk.mean_fidelity, s=14, alpha=0.7)
    lower = min(gate_risk.nominal_fidelity.min(), gate_risk.mean_fidelity.min())
    plt.plot([lower, 1], [lower, 1], "--", lw=1)
    plt.xlabel("Nominal fidelity")
    plt.ylabel("Mean perturbed fidelity")
    plt.grid(alpha=0.2)
    save_figure(f"joint_{profile_name}_nominal_vs_mean")

    plt.figure(figsize=(7, 4.6))
    plt.hist(gate_risk.worst_fidelity, bins=35, alpha=0.8, label="worst scenario")
    plt.hist(1 - gate_risk.cvar10_infidelity, bins=35, alpha=0.55, label="1 - CVaR10 infidelity")
    plt.xlabel("Gate-level robustness score")
    plt.ylabel("Gate count")
    plt.legend()
    plt.grid(alpha=0.15)
    save_figure(f"joint_{profile_name}_tail_metrics")

for array, label in [(mean_map, "mean"), (p05_map, "p05")]:
    plt.figure(figsize=(7.2, 5.4))
    image = plt.imshow(
        array,
        origin="lower",
        aspect="auto",
        extent=[rf_values[0] * 100, rf_values[-1] * 100, b0_values[0], b0_values[-1]],
        vmin=max(0, float(array.min())),
        vmax=1,
    )
    plt.colorbar(image, label=f"{label} fidelity")
    plt.xlabel("RF gain error (%)")
    plt.ylabel("Common B0 / carrier offset (Hz)")
    plt.title(f"{label.upper()} fidelity: B0 × RF gain")
    save_figure(f"b0_rf_gain_{label}_heatmap")

In [ ]:
report = {
    "checkpoint_step": 48600,
    "checkpoint_stats": reference_summary,
    "checkpoint_sha256": sha256_file(CHECKPOINT_PATH),
    "physics": {
        "v_hz": list(CFG.v_hz),
        "J_hz": list(CFG.j_hz),
        "dt_s": CFG.dt_s,
        "n_slices": CFG.n_slices,
        "total_pulse_s": CFG.dt_s * CFG.n_slices,
        "interaction": "full XX+YY+ZZ",
    },
    "new_nominal": nominal_summary,
    "profiles": PROFILES,
    "sample_counts": {
        "new_haar_gates": N_NEW_GATES,
        "sweep_gates": N_SWEEP_GATES,
        "map_gates": N_MAP_GATES,
        "joint_scenarios_per_profile": N_JOINT_SCENARIOS,
    },
    "stress_seed": STRESS_SEED,
}

with (OUTPUT_ROOT / "stress_test_report.json").open("w") as handle:
    json.dump(report, handle, indent=2, default=str)

with (OUTPUT_ROOT / "environment.json").open("w") as handle:
    environment = {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    json.dump(environment, handle, indent=2)

with (OUTPUT_ROOT / "MANIFEST.txt").open("w") as handle:
    handle.write("\n".join(
        str(path.relative_to(OUTPUT_ROOT))
        for path in sorted(OUTPUT_ROOT.rglob("*"))
        if path.is_file()
    ))

checksum_entries = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and path.name != "SHA256SUMS.txt":
        checksum_entries.append(f"{sha256_file(path)}  {path.relative_to(OUTPUT_ROOT)}")

with (OUTPUT_ROOT / "SHA256SUMS.txt").open("w") as handle:
    handle.write("\n".join(checksum_entries))

print("Stress-test outputs:", OUTPUT_ROOT)
print("Files written:", sum(1 for path in OUTPUT_ROOT.rglob("*") if path.is_file()))
display(joint_summary)